# tabular_agent — Phase 3, step 6: the run reads its own history

Step 5 gave the agent a way to look at **one experiment's rows**. This one gives it
a way to look at **every experiment at once** — and it is the smaller, stranger tool
of the two, because it never sees a row of data at all.

The problem it solves is arithmetic by eye. By experiment twenty the agent has read
twenty result dicts, one at a time, each still sitting in its context, and the
question "did target encoding actually pay?" is answered by impression. It is the
same failure `0003` fixed for *data* — reasoning over numbers-as-text, with the
arithmetic done by vibes — one level up, over the run's own records.

`explore_journal` hands those records to the sandbox as DataFrames:

    lb      one row per experiment
    tables  one row per feature table
    notes   the run log
    run     the scalars: task, split, stop, best, baseline, higher_is_better

`lb.groupby("target_transform")["val_score"].mean()` is exact, and the nineteen
experiments not currently being thought about stay on disk instead of in the
context window.

**What to watch for:**

1. it needs **no leakage argument** — the thing that made step 5 delicate is simply
   absent here, and §8 shows why
2. the columns are **declared**, not inferred, so a run that has trained nothing
   still hands over a well-formed frame
3. the queries in §4–§7 answer questions **no single training result can**, which is
   the entire justification for the tool
4. §10: what this is really for is not tidiness, it is **episodes** — a resumed
   agent with no memory has to rebuild its picture of the run from somewhere

> Ships unexecuted. Every code cell was smoke-tested in order first; run it top to
> bottom and the numbers in the prose should be the numbers you see.

## 1. A run worth querying

Apartments again, but built so that several *different ideas* are on the table and
the leaderboard has something to say about which of them paid:

- `district` and `condition` are strings, so training drops them — `fit_encoding`
  is the only way in, and there are three methods to choose between
- `condition` is genuinely **ordered** (`poor < fair < good < excellent`), so
  `ordinal` is defensible where it would be nonsense for `district`
- the price is multiplicative and lognormal-ish (skew ≈ 1.6), so a **target
  transform** should help, and `log1p` versus `none` is a real comparison rather
  than a decoration

In [ ]:
import os, shutil, json
import numpy as np, pandas as pd

WORK = os.path.abspath("_step6_demo")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

rng = np.random.default_rng(7)
n = 800

area = rng.uniform(30, 200, n)
rooms = rng.integers(1, 6, n)
district = rng.choice(["altstadt", "hafen", "vorstadt", "gruenau"], n, p=[.2, .25, .35, .2])
condition = rng.choice(["poor", "fair", "good", "excellent"], n, p=[.15, .35, .35, .15])

level = {"altstadt": 2600, "hafen": 1600, "vorstadt": 950, "gruenau": 700}
cond_mult = {"poor": 0.75, "fair": 0.9, "good": 1.05, "excellent": 1.3}

# Multiplicative, with lognormal noise: the shape that makes a log target
# transform a real experiment rather than a stylistic preference.
price = (
    area * np.array([level[d] for d in district])
    * np.array([cond_mult[c] for c in condition])
    + rooms * 5000
) * np.exp(rng.normal(0, 0.18, n))

df = pd.DataFrame({"area": area, "rooms": rooms, "district": district,
                   "condition": condition, "price": price})
data_path = os.path.join(WORK, "apartments.csv")
df.to_csv(data_path, index=False)

print("target skew:", round(float(df["price"].skew()), 2))
df.head()

`inspect_data` first, as always: it pins the task type and the metric, and names the
columns the models cannot use as they stand.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness import tools
from tabular_agent.taskspec import TaskSpec

session = RunSession(os.path.join(WORK, "run"), data_path, TaskSpec(target="price"))
info = tools.inspect_data(session)

print("metric:", info["metric"], "| higher_is_better:", info["higher_is_better"])
print("dropped as non-numeric:", info["non_numeric_columns"])

## 2. The first query, before anything has been trained

Start here, because it is the case that breaks a tool built the obvious way.

`lb` is assembled from a **declared** column list (`_LB_COLUMNS`), not inferred from
whatever rows happen to exist. An empty run therefore hands over a frame with zero
rows and all the right columns, and `lb[lb["val_score"] < 0.2]` returns nothing
rather than raising `KeyError`.

That is not a corner case. "What has been done so far?" is the *first* thing a
resumed agent asks, and on a fresh run the honest answer is "nothing" — which has to
arrive as an answer, not as a crash the agent then tries to debug.

In [ ]:
tools.explore_journal(session, '''
def explore(lb):
    return {"rows": len(lb),
            "columns": list(lb.columns),
            "filtered": len(lb[lb["val_score"] < 0.2])}
''')

The contract is the same as `explore_experiment`'s: define `explore(...)`, name the
frames you want as parameters, return something small and JSON-shaped. The namespace
plumbing is the one built in step 5 §2 — this tool is what it was built for.

`run` is the frames' companion: the scalars a DataFrame cannot hold, and without
which the frames cannot be read correctly. `higher_is_better` is the load-bearing
one — see §7.

In [ ]:
out = tools.explore_journal(session, '''
def explore(lb, tables, notes, run):
    print("shapes:", lb.shape, tables.shape, notes.shape)
    return {"run_keys": sorted(run), "task": run["task"],
            "higher_is_better": run["higher_is_better"],
            "split": run["split"]}
''')
print(out["stdout"])
out["result"]

## 3. Ten experiments, four ideas

Now give it something to query. Three encodings, a derived ratio, both target
transforms, two model sizes — driven by hand here, but this is exactly the sequence
a run produces on its own.

In [ ]:
P = {"n_estimators": 80}

def train(**kw):
    out = tools.train_tabular_model(session, **kw)
    print(f"{out['experiment_id']:>3}  table={str(kw.get('table')):<20}"
          f"  transform={kw.get('target_transform', 'none'):<6}"
          f"  val={out['val_score']:.4f}  cv={out['cv_score']:.4f}")
    return out

train(model_type="baseline")                                   # the floor
train(model_type="lgbm", params=P)                             # raw columns only
train(model_type="lgbm", params=P, target_transform="log1p")

# Three ways to bring the two string columns in. `ordinal` states the level order —
# it is never inferred, which is why it is only offered `condition`.
freq = tools.fit_encoding(session, ["district", "condition"], "frequency")
tgt  = tools.fit_encoding(session, ["district", "condition"], "target")
ordn = tools.fit_encoding(session, ["condition"], "ordinal",
                          categories={"condition": ["poor", "fair", "good", "excellent"]})

train(table=freq["table"], model_type="lgbm", params=P)
train(table=freq["table"], model_type="lgbm", params=P, target_transform="log1p")
train(table=tgt["table"],  model_type="lgbm", params=P)
train(table=tgt["table"],  model_type="lgbm", params=P, target_transform="log1p")
train(table=ordn["table"], model_type="lgbm", params=P, target_transform="log1p")

ratio = tools.transform_features(session, '''
def transform_features(df):
    df = df.copy()
    df["area_per_room"] = df["area"] / df["rooms"]
    return df
''', table=tgt["table"])

train(table=ratio["table"], model_type="lgbm", params=P, target_transform="log1p")
train(table=ratio["table"], model_type="lgbm", params={"n_estimators": 200},
      target_transform="log1p")

# And one idea started but not finished: a table built, and no experiment run on
# it. This is what the end of an episode looks like when the cut lands mid-idea.
tools.transform_features(session, '''
def transform_features(df):
    df = df.copy()
    df["sqrt_area"] = df["area"] ** 0.5
    return df
''', table=freq["table"])

### The problem, stated in characters

Ten experiments is where reading the leaderboard as text stops being pleasant and
starts being expensive. The agent has already paid for all of it — every one of
those result dicts is in its context — and it will pay again on every turn.

In [ ]:
from tabular_agent.harness import journal

j = journal.load(session.run_dir)
as_text = json.dumps(j["leaderboard"])
print(f"{len(j['leaderboard'])} experiments = {len(as_text):,} chars"
      f"  (~{len(as_text)//4:,} tokens, very roughly)")
print(f"extrapolated to 150 experiments: ~{len(as_text)//4 * 15:,} tokens")

answer = tools.explore_journal(session, '''
def explore(lb, run):
    best = lb["val_score"].idxmin() if not run["higher_is_better"] else lb["val_score"].idxmax()
    return lb.loc[best, ["id", "source_table", "target_transform", "val_score"]].to_dict()
''')
print("\nthe same run, answered:", len(json.dumps(answer))//4, "tokens")
answer["result"]

That ratio is the whole argument. Reciting history scales with the run; **querying**
it scales with the question. At 150 experiments the journal is 60–120KB, reciting it
costs 15–30k tokens, and there is no context budget in which that is a sensible way
to remember what you did.

## 4. The question no single result can answer

Which encoding method actually paid? Nothing in any one training result knows this —
each result knows its own score and, at best, its gain over its source table. The
answer lives *across* rows, and it comes from a join: `lb.source_table` is a path,
`tables.path` is the same path, and `tables.method` is what made it.

In [ ]:
tools.explore_journal(session, '''
def explore(lb, tables):
    m = lb.merge(tables, left_on="source_table", right_on="path", how="left")
    m["method"] = m["method"].fillna("raw input table")
    return m.groupby("method")["val_score"].agg(["min", "mean", "size"]).round(4).to_dict("index")
''')

Read it as the agent would. `target` (0.2092) and `frequency` (0.2201) are within a
whisker of each other and both are worth about **0.27 rmsle** against the best raw
score of 0.4883. `ordinal` bought **0.025** — an order of magnitude less.

The lesson is not "ordinal is bad". `ordinal` was the only method offered a single
column, because a level order for `district` would be an invention; so it left
`district` a dropped string and the model blind to the dominant term in the
generating rule. The encoding *method* mattered far less than **which columns got
encoded at all** — and one `groupby` says so, while ten result dicts read one at a
time do not.

## 5. The controlled pair: did the target transform pay?

`target_transform` (`none` / `log1p` / `sqrt`) is a per-experiment argument on
`train_tabular_model` — deliberately, per `0021`. A project-level setting would be an
assumption; an argument makes it an **experiment**, and an experiment ends up as a
column in `lb` that can be grouped.

The naive version of the question is a marginal mean over the whole leaderboard. Run
it first, because the number it produces is wrong in an instructive way.

In [ ]:
tools.explore_journal(session, '''
def explore(lb):
    return lb.groupby("target_transform")["val_score"].agg(["mean", "min", "size"]).round(4).to_dict("index")
''')

`log1p` mean 0.302 against `none` 0.4026 — a gap of **0.10**, which would be a
spectacular result. It is an artifact. `log1p` was tried six times and mostly on the
*good* tables; `none` was tried four times and half of those were on the raw table,
where every model is bad. The grouping is measuring which tables each transform
happened to be paired with.

The honest version **holds the table constant** and compares within it. That is a
pivot, and it is the kind of thing that is tedious by eye and trivial in three lines:

In [ ]:
tools.explore_journal(session, '''
def explore(lb):
    paired = lb[lb["model_type"] == "lgbm"].pivot_table(
        index="source_table", columns="target_transform", values="val_score")
    paired = paired.dropna()
    paired["log1p_gain"] = paired["none"] - paired["log1p"]   # rmsle: lower is better
    return paired.round(4).to_dict("index")
''')

Three tables were trained both ways and `log1p` wins on all three — but by **0.005 to
0.019**, not 0.10. The marginal mean overstated it by roughly an order of magnitude.

(`data/work.parquet` in that index is the raw table: the leaderboard names the work
half by its path, and `table=None` resolves to it.)

The pattern in the honest numbers is the interesting one: the gain is ~4x larger on
the encoded tables (0.019, 0.013) than on the raw one (0.005). Once the model can see
`district`, the multiplicative structure of the target is what is left to get wrong,
and that is exactly what a log transform addresses. That is a real finding about
*this* dataset, and it took one query over history rather than a fourth experiment.

Note what made the pivot possible: the transform is recorded per row. A setting
chosen once at the top of the run would leave no column to group by, and the question
would be unanswerable from the journal — which is `0021`'s argument arriving from the
other end.

## 6. `params` is a JSON string, on purpose

The obvious question about hyper-parameters is `lb.groupby("params")`. With a dict in
the cell that raises `TypeError: unhashable type: 'dict'` — the *one* query the column
exists for, refused by pandas.

So `_journal_frames` serialises it with `json.dumps(..., sort_keys=True)`. It groups,
it sorts, it compares by equality, and it is one `json.loads` away from the values.

In [ ]:
tools.explore_journal(session, '''
def explore(lb):
    import json
    d = lb[lb["model_type"] == "lgbm"].copy()
    d["n_estimators"] = d["params"].map(lambda s: json.loads(s)["n_estimators"])

    marginal = d.groupby("n_estimators")["val_score"].agg(["min", "size"]).round(4)
    # The same confound as §5, so the only claim worth making is the paired one:
    # the tables where both settings were actually tried.
    paired = d.pivot_table(index="source_table", columns="n_estimators",
                           values="val_score").dropna().round(4)
    return {"marginal": marginal.to_dict("index"), "paired": paired.to_dict("index")}
''')

`params` grouped, which is the point of the section. What it grouped *into* is the §5
trap again — the 80-tree group's best score comes from a table the 200-tree setting was
never tried on. The only honest comparison is the paired one: on `t4`, 200 trees scored
**0.2200** against 80 trees' **0.2109**. More trees did not help.

Worth seeing, because it is the cheapest possible finding and the agent would otherwise
spend experiments rediscovering it. `table_patience` (`0014`) exists for exactly this
reason: hyper-parameter sweeps on a fixed table cost the same as a feature experiment
and tell you far less.

## 7. Two traps the frames do not protect you from

**Direction.** Most metrics here are losses, so `max()` is the *worst* model. Nothing
in `lb` says which way is better — `run["higher_is_better"]` does, and it is in the
namespace for this reason alone. The tool docstring says so out loud, because this is
a mistake an LLM makes fluently and silently.

In [ ]:
tools.explore_journal(session, '''
def explore(lb, run):
    hib = run["higher_is_better"]
    right = lb.loc[lb["val_score"].idxmax() if hib else lb["val_score"].idxmin()]
    wrong = lb.loc[lb["val_score"].idxmin() if hib else lb["val_score"].idxmax()]
    return {"metric": run["task"]["metric"], "higher_is_better": hib,
            "best": [right["id"], round(right["val_score"], 4)],
            "if_you_forget": [wrong["id"], round(wrong["val_score"], 4)]}
''')

**Two scores, and the gap between them is the point.** `cv_score` is measured on the
folds the agent built features against; `val_score` is measured on the held-out half
the leaderboard ranks by (`0019`). Their difference is this run's own record of how
much of each gain lived only in the folds.

In [ ]:
tools.explore_journal(session, '''
def explore(lb):
    d = lb[lb["model_type"] == "lgbm"].copy()
    d["optimism"] = d["val_score"] - d["cv_score"]     # rmsle: positive = val is worse
    cols = ["id", "source_table", "cv_score", "val_score", "optimism", "fold_spread"]
    return d[cols].round(4).to_dict("records")
''')

Read the sign carefully. The weak experiments — raw columns, and the ordinal table that
never saw `district` — have *negative* optimism: validation came out slightly better
than the folds, which is the two halves simply differing. Every strong model has
**positive** optimism of 0.010–0.019, or 5–9% of its own score.

Resist the obvious explanation. It is tempting to blame target encoding, but the
largest gap (0.0193) belongs to a **frequency**-encoded table, which never looks at the
target at all. The pattern that actually fits is duller and more useful: the better a
model fits the work half, the more of that fit fails to transfer.

Now put it next to `fold_spread`, which is on the same rows: ~0.009. The gap between
this run's two scores is **larger than the fold-to-fold wobble used to judge whether an
improvement is real** — and the improvement bar is a quarter of that wobble, ~0.002.
That is `0022`'s finding in miniature. On house-prices our validation score said a
target transform bought 0.008 and Kaggle said 0.0007: 91% optimism, one split further
out. The value of having it in `lb` is that a run can compute this about itself while
it still has time to act on it.

### And the question that is pure bookkeeping

Which tables were built and never trained on? Trivially expressible, and it is the
question a **resumed** agent most needs answered — it did not build them and has no
memory that it meant to.

In [ ]:
tools.explore_journal(session, '''
def explore(lb, tables):
    trained = set(lb["source_table"])
    idle = tables[~tables["path"].isin(trained)]
    return {"tables": len(tables), "never_trained": idle["path"].tolist(),
            "columns_added": {r["path"]: r["n_columns"] for _, r in idle.iterrows()}}
''')

`t5` is the table §3 built and abandoned — one column, no experiment. `0024` calls
episode length "a seam, not a count": an idea spans several calls (build a table,
train it, extend it, train again) and an episode that ends in the middle of one leaves
exactly this trace. Without the query, the next agent either re-derives the idea or
never finds it. With it, "finish what the last episode started" is three lines.

## 8. Why this one needs no leakage argument

Step 5 needed a careful one: `explore_experiment` hands over real rows, including the
target, and the defence is that there is no path to persist anything computed from
them.

The obvious explanation for the difference is "nothing in this namespace is a row",
and it is true — `lb`, `tables`, `notes` and `run` hold only values a tool produced or
the agent wrote. But it is the *easy* half, and taking it for the whole answer is a
mistake worth making explicitly, because nobody was ever going to put `oof_pred` in a
frame called "the run's own records". Keeping rows out costs no judgement.

**The exclusion that costs judgement is `profiles` from `inspect_data`.** A profile
looks exactly like metadata: it is small, it is a summary, it was produced by a tool
the agent called, and every argument for putting `lb` in this namespace applies to it
word for word.

The catch is that a profile carries **target-conditional statistics** — a per-column
correlation with the target, computed over all rows. Let the agent query that freely
and it is doing feature selection on the full dataset. Every data-path guard still
holds: no encoder was fitted wrong, no row crossed a fold boundary,
`transform_features` never saw the target. The leakage arrives down the **metadata**
path, which is the one path with no guard on it, because the leak runs through the
model's reasoning rather than through the data.

So the honest version of this section's claim is narrower than "it sees no rows":
*distributional facts about features are fine; anything conditioned on the target is
not* — and that line has to be drawn per field, not per frame. It is why the parked
metadata-kernel sketch says the profile would have to be **split** before any of it
could go in.

The frame names are the whole surface, and asking for one that does not exist gets you
the list rather than a `KeyError` from somewhere deeper:

In [ ]:
from tabular_agent.sandbox import SandboxError

try:
    tools.explore_journal(session, "def explore(df):\n    return len(df)")
except SandboxError as e:
    print(e)

## 9. Free, silent, and available after the end

Same deal as `explain_table` and `explore_experiment`: no leaderboard row, no budget
spent, no journal write. Charging for a look at history would price *understanding the
run* the same as *extending it*, and the agent would ration the wrong one.

There is also no kernel — a name bound in one call is gone by the next (`0020`). The
state is the run directory, so a resumed agent must see exactly what this one sees.

In [ ]:
before = json.dumps(journal.load(session.run_dir), sort_keys=True)

for _ in range(3):
    tools.explore_journal(session, "def explore(lb):\n    return len(lb)")

print("journal unchanged after 3 calls:",
      json.dumps(journal.load(session.run_dir), sort_keys=True) == before)

try:
    tools.explore_journal(session, "def explore(lb):\n    global stash\n    stash = lb\n    return 'stashed'")
    tools.explore_journal(session, "def explore(lb):\n    return len(stash)")
except SandboxError as e:
    print("no kernel:", str(e).splitlines()[0])

And it works after `finalize_run` — which is when "what did this run actually learn"
stops being a tactical question and becomes the only one left.

In [ ]:
report = tools.finalize_run(session)
print("best:", report["best_experiment_id"], "| val:", round(report["val_score"], 4))

tools.explore_journal(session, '''
def explore(lb, run):
    return {"finalized": run["stop"]["finalized"],
            "experiments": len(lb),
            "best": run["best"],
            "baseline": run["baseline"]}
''')

## 10. What it is actually for: an agent that has forgotten everything

Everything above works at ten experiments, and at ten experiments none of it is
necessary — the leaderboard is cheaper to read as text than to query. The
justification is the long run, and the long run needs **episodes**: a driver loop
where each iteration is a fresh agent on the same `run_dir`, so context resets instead
of climbing (`0024`).

A fresh agent wakes up knowing nothing. `explore_journal` is how it finds out, and the
live runs said two things about that which are worth carrying:

**It is discoverable.** The v5 resume prompt said only "earlier episodes did things
you have no memory of" — no counts, no scores, and **no mention of the tool**. Every
resumed episode opened `inspect_data → explore_journal → explore_journal`. The
docstring alone was enough, which is a good reason to read that docstring as prompt
text rather than as documentation.

**It is not free in the way that matters.** Orientation ate roughly **a third of every
episode** at both caps tested, and one episode spent seven `explore_journal` calls and
trained nothing (`0025`). Given more room the agent orients *more*, not less. So the
tool does not replace a derived brief — the brief is rendered from the journal, is
fixed-size forever, and arrives without costing a tool call. They are complementary,
and the brief is the outstanding piece.

In [ ]:
from tabular_agent.harness.agent import build_agent

agent = build_agent("openai:gpt-4o")
print("the surface, now eight tools:")
for name in sorted(agent._function_toolset.tools):
    print(" -", name)

### Two things writing this notebook found

**A wrong answer, now fixed.** The `tables` frame built `columns` and `n_columns` from
a `columns` key that journal rows do not have — they carry `new_columns` — so every
table reported **zero columns added**, for every run, since the tool was written. Not
a crash: a confident zero, to the one question that frame exists to answer. It was
invisible until §7 asked the question with an expected answer in hand, which is the
argument for notebooks over unit tests in one line. Fixed, with a test
(`test_a_table_reports_the_columns_it_added`).

**A trap still standing.**

`notes` is empty on every run in this repo, and unlike `lb` and `tables` its columns
are **not** declared — it is built straight from `journal["log"]`, which no agent-facing
tool writes today. `journal.append_note` exists; nothing calls it.

So on a fresh run `notes` is a `(0, 0)` frame, and `notes["hypothesis"]` raises a
`KeyError` rather than returning nothing — precisely the failure mode §2 was careful to
avoid for `lb`. Harmless while the frame is always empty, and a trap the day a note
tool lands.

In [ ]:
tools.explore_journal(session, '''
def explore(notes):
    return {"shape": list(notes.shape), "columns": list(notes.columns)}
''')

## 11. Handing it to a model

Everything above was driven by hand on purpose. The tool is registered, so a live agent
can ask its own questions — and the thing to watch in the transcript is *when* it
reaches for this: after a disappointing score (good), before deciding what to try next
(better), or seven times in a row while training nothing (the failure mode in `0025`).

Needs `config.env` with a `REQUESTY_API_KEY`. In a notebook use `await agent.run(...)`
— `run_sync` raises inside a running event loop.

In [ ]:
from tabular_agent.harness.agent import build_agent, gateway_model, DEFAULT_GATEWAY_MODEL, Deps

PROMPT = """Build the best model you can for this dataset. Before you finalize,
work out from the run's own history which of the ideas you tried actually paid, and
say so in your summary."""

live = RunSession(os.path.join(WORK, "run_llm"), data_path, TaskSpec(target="price"))
agent = build_agent(gateway_model(DEFAULT_GATEWAY_MODEL))

result = await agent.run(PROMPT, deps=Deps(session=live))
print(result.output)

### Where this leaves the sequence

The two halves of the parked "metadata kernel" idea from 2026-08-06 are now both
built — `explore_experiment` over one experiment's rows, `explore_journal` over the
run's records — and they share one sandbox and one read-only path.

What is still outstanding is the piece §10 argues for: the **derived brief**, rendered
from the journal into the episode's system prompt, so a resumed agent starts oriented
instead of paying a third of its episode to become so. After that, moving the episode
loop out of `scripts/` and into `harness/`, and `val_se` (`0022`) so a run can tell
that a gain is inside the noise while it still has time to act on it.